# Evaluating Third-Party Agent Trajectories with NeMo Agent Toolkit

This notebook demonstrates **ATIF as an interoperability contract** between
third-party agent frameworks and the NeMo Agent Toolkit evaluation harness.

We take real ATIF trajectories generated by
[Harbor](https://github.com/harbor-framework/harbor) running the
`mini-swe-agent` on [BFCL](https://gorilla.cs.berkeley.edu/leaderboard.html)
(Berkeley Function-Calling Leaderboard) tasks, then evaluate them
through `nvidia-nat-eval` using both custom and RAGAS evaluators.

```
Harbor CLI  -->  ATIF JSON  -->  NAT ATIF Models  -->  EvaluationHarness  -->  Results
 (any agent)     (standard)      (Pydantic parse)      (custom + RAGAS)
```

**What this proves:**
- Any agent framework that outputs ATIF trajectories can be evaluated by NeMo Agent Toolkit
- Both lightweight custom evaluators and LLM-as-judge evaluators work on the same ATIF data
- No NeMo Agent toolkit workflow is needed — `nvidia-nat-eval` is a standalone component

**Requirements:**
- `nvidia-nat-eval` (core eval harness + custom evaluators)
- `nvidia-nat-ragas` (RAGAS evaluator plugin, for Section 6)
- `NVIDIA_API_KEY` environment variable (for RAGAS evaluators only)

## 1. Install Dependencies

In [ ]:
# Install nat-eval, RAGAS evaluator plugin, and langchain (needed by RAGAS for LLM wrapping).
#
# For development on this branch, install from local source:
!uv pip install -q \
    -e ../../packages/nvidia_nat_eval \
    -e ../../packages/nvidia_nat_ragas \
    -e ../../packages/nvidia_nat_langchain
#
# For released versions, use:
# !uv pip install nvidia-nat-eval nvidia-nat-ragas nvidia-nat-langchain

## 2. How These Trajectories Were Generated (Harbor CLI)

The ATIF trajectories used in this notebook were generated using the
[Harbor](https://github.com/harbor-framework/harbor) framework, which runs
agents against benchmark datasets and outputs ATIF-formatted trajectory logs.

### Install Harbor

```bash
uv tool install harbor
```

### Set Environment Variables

```bash
export NVIDIA_NIM_API_KEY="$NVIDIA_API_KEY"
export MSWEA_COST_TRACKING='ignore_errors'
```

### Run mini-swe-agent on BFCL

```bash
harbor run --dataset bfcl \
    --agent mini-swe-agent \
    --model meta/llama-3.3-70b-instruct \
    --n-tasks 5
```

Harbor writes results to `jobs/<timestamp>/<task-name>/agent/trajectory.json`.
Each `trajectory.json` is a valid ATIF document that can be loaded directly
into the NeMo Agent Toolkit ATIF models.

**Note:** Running Harbor requires Docker. The trajectories below are
pre-bundled from an actual run so this notebook works without Docker.

## 3. Load Pre-Bundled ATIF Trajectories

These are real ATIF trajectories produced by Harbor's `mini-swe-agent` on BFCL
tasks using `meta/llama-3.3-70b-instruct` via NVIDIA NIM. Each task gives the agent
a natural language request and a set of available function signatures; the agent
must determine which function(s) to call (or output `[]` if none apply).

We include three trajectories with different outcomes:

| Task | Description | Harbor Reward |
|---|---|---|
| `bfcl-live-simple` | Simple function call (`ChaFod`) | 1.0 (correct) |
| `bfcl-live-irrelevance` | No applicable function | 1.0 (correct) |
| `bfcl-live-multiple` | Multi-function selection | 0.0 (incorrect) |

In [ ]:
trajectory_simple = {
    "schema_version": "ATIF-v1.2",
    "session_id": "bb70227e-9b5c-4eb4-800d-cba8ce47c2b5",
    "agent": {
        "name": "mini-swe-agent",
        "version": "2.2.7",
        "model_name": "nvidia_nim/meta/llama-3.3-70b-instruct",
    },
    "steps": [
        {
            "step_id": 1,
            "timestamp": "2026-03-12T23:41:46.293396+00:00",
            "source": "system",
            "message": "You are a helpful assistant that can interact with a computer.",
        },
        {
            "step_id": 2,
            "timestamp": "2026-03-12T23:41:46.293440+00:00",
            "source": "user",
            "message": (
                "I need Whopper also known old folks as the burger.\n\n"
                "## Available Functions\n\n"
                "### ChaFod\n\n"
                "**Description:** Changes the selection of food based on the customer's request, "
                "ensuring the food name provided is in uppercase as per the requirement.\n\n"
                "**Parameters:**\n"
                "- `TheFod` (string, Required): The name of the food to be changed, "
                "provided in uppercase letters only (e.g., 'PIZZA', 'BURGER')."
            ),
        },
        {
            "step_id": 3,
            "timestamp": "2026-03-12T23:41:46.293718+00:00",
            "source": "agent",
            "model_name": "nvidia_nim/meta/llama-3.3-70b-instruct",
            "message": "",
            "tool_calls": [
                {
                    "tool_call_id": "chatcmpl-tool-9ae54b6047a8b8d2",
                    "function_name": "bash",
                    "arguments": {
                        "command": "echo '[{\"ChaFod\": {\"TheFod\": \"BURGER\"}}]' > /app/result.json"
                    },
                }
            ],
            "observation": {
                "results": [{"content": '{\n  "returncode": 0,\n  "output": ""\n}'}]
            },
            "metrics": {"prompt_tokens": 2429, "completion_tokens": 45},
        },
        {
            "step_id": 4,
            "timestamp": "2026-03-12T23:41:46.293744+00:00",
            "source": "agent",
            "model_name": "nvidia_nim/meta/llama-3.3-70b-instruct",
            "message": "",
            "tool_calls": [
                {
                    "tool_call_id": "chatcmpl-tool-9179d9a42c4057fc",
                    "function_name": "bash",
                    "arguments": {"command": "echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT"},
                }
            ],
            "observation": {
                "results": [
                    {
                        "content": (
                            '{"returncode": -1, "output": "",'
                            ' "exception_info": "action was not executed"}'
                        )
                    }
                ]
            },
            "metrics": {"prompt_tokens": 2496, "completion_tokens": 29},
        },
    ],
    "notes": "Converted from mini-swe-agent trajectory format to ATIF",
    "final_metrics": {"total_prompt_tokens": 4925, "total_completion_tokens": 74},
}

In [ ]:
trajectory_irrelevance = {
    "schema_version": "ATIF-v1.2",
    "session_id": "efe0fd81-5837-4ce5-8c43-ea0fa071adf3",
    "agent": {
        "name": "mini-swe-agent",
        "version": "2.2.7",
        "model_name": "nvidia_nim/meta/llama-3.3-70b-instruct",
    },
    "steps": [
        {
            "step_id": 1,
            "timestamp": "2026-03-12T23:40:18.052638+00:00",
            "source": "system",
            "message": "You are a helpful assistant that can interact with a computer.",
        },
        {
            "step_id": 2,
            "timestamp": "2026-03-12T23:40:18.052697+00:00",
            "source": "user",
            "message": (
                "Call me an Uber ride type \"Plus\" in Berkeley at zipcode 94704 in 10 minutes\n\n"
                "## Available Functions\n\n"
                "### user.mandates\n\n"
                "**Description:** Fetches the mandates for a user given the user's ID "
                "and the status of the mandate.\n\n"
                "**Parameters:**\n"
                "- `user_id` (string, Required): The unique identifier of the user.\n"
                "- `status` (string, Required): The status of the mandates to be fetched."
            ),
        },
        {
            "step_id": 3,
            "timestamp": "2026-03-12T23:40:18.052899+00:00",
            "source": "agent",
            "model_name": "nvidia_nim/meta/llama-3.3-70b-instruct",
            "message": "",
            "tool_calls": [
                {
                    "tool_call_id": "chatcmpl-tool-81444756b227dd8b",
                    "function_name": "bash",
                    "arguments": {"command": "ls -la"},
                },
                {
                    "tool_call_id": "chatcmpl-tool-946762a27f016fdb",
                    "function_name": "bash",
                    "arguments": {"command": "echo '[]' > /app/result.json"},
                },
                {
                    "tool_call_id": "chatcmpl-tool-998c220d5a83d7cd",
                    "function_name": "bash",
                    "arguments": {"command": "echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT"},
                },
            ],
            "observation": {
                "results": [
                    {"content": '{\n  "returncode": 0,\n  "output": "total 12\\n...\\n"\n}'},
                    {"content": '{\n  "returncode": 0,\n  "output": ""\n}'},
                    {
                        "content": (
                            '{"returncode": -1, "output": "",'
                            ' "exception_info": "action was not executed"}'
                        )
                    },
                ]
            },
            "metrics": {"prompt_tokens": 1150, "completion_tokens": 182},
        },
    ],
    "notes": "Converted from mini-swe-agent trajectory format to ATIF",
    "final_metrics": {"total_prompt_tokens": 1150, "total_completion_tokens": 182},
}

In [ ]:
trajectory_multiple = {
    "schema_version": "ATIF-v1.2",
    "session_id": "5e1aa9d3-6e0c-4f9b-9fe4-0051324c6dfc",
    "agent": {
        "name": "mini-swe-agent",
        "version": "2.2.7",
        "model_name": "nvidia_nim/meta/llama-3.3-70b-instruct",
    },
    "steps": [
        {
            "step_id": 1,
            "timestamp": "2026-03-12T23:45:12.779999+00:00",
            "source": "system",
            "message": "You are a helpful assistant that can interact with a computer.",
        },
        {
            "step_id": 2,
            "timestamp": "2026-03-12T23:45:12.780028+00:00",
            "source": "user",
            "message": (
                "Find me a museum to visit in New York? I prefer it to be free for entry.\n\n"
                "## Available Functions\n\n"
                "### Travel_1_FindAttractions\n\n"
                "**Description:** Retrieves a list of attractions within a specified city, "
                "filtered by entry fee, category, and suitability for children.\n\n"
                "**Parameters:**\n"
                "- `location` (string, Required): City in 'City, State' format.\n"
                "- `free_entry` (string, Optional): 'True', 'False', or 'dontcare'.\n"
                "- `category` (string, Optional): e.g. 'Museum', 'Park', or 'dontcare'.\n"
                "- `good_for_kids` (string, Optional): 'True', 'False', or 'dontcare'."
            ),
        },
        {
            "step_id": 3,
            "timestamp": "2026-03-12T23:45:12.780545+00:00",
            "source": "agent",
            "model_name": "nvidia_nim/meta/llama-3.3-70b-instruct",
            "message": "",
            "tool_calls": [
                {
                    "tool_call_id": "chatcmpl-tool-bf0a8b0c889fa555",
                    "function_name": "bash",
                    "arguments": {
                        "command": (
                            'echo [{"Travel_1_FindAttractions": '
                            '{"location": "New York, NY", "free_entry": "True", '
                            '"category": "Museum", "good_for_kids": "dontcare"}}] '
                            "> /app/result.json"
                        )
                    },
                }
            ],
            "observation": {
                "results": [{"content": '{\n  "returncode": 0,\n  "output": ""\n}'}]
            },
            "metrics": {"prompt_tokens": 5908, "completion_tokens": 72},
        },
        {
            "step_id": 4,
            "timestamp": "2026-03-12T23:45:12.780575+00:00",
            "source": "agent",
            "model_name": "nvidia_nim/meta/llama-3.3-70b-instruct",
            "message": "",
            "tool_calls": [
                {
                    "tool_call_id": "chatcmpl-tool-876670e5c4969598",
                    "function_name": "bash",
                    "arguments": {"command": "echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT"},
                }
            ],
            "observation": {
                "results": [
                    {
                        "content": (
                            '{"returncode": -1, "output": "",'
                            ' "exception_info": "action was not executed"}'
                        )
                    }
                ]
            },
            "metrics": {"prompt_tokens": 6002, "completion_tokens": 30},
        },
    ],
    "notes": "Converted from mini-swe-agent trajectory format to ATIF",
    "final_metrics": {"total_prompt_tokens": 11910, "total_completion_tokens": 102},
}

In [ ]:
from nat.data_models.atif import ATIFTrajectory

trajectories = {
    "bfcl-live-simple": ATIFTrajectory.model_validate(trajectory_simple),
    "bfcl-live-irrelevance": ATIFTrajectory.model_validate(trajectory_irrelevance),
    "bfcl-live-multiple": ATIFTrajectory.model_validate(trajectory_multiple),
}

for name, traj in trajectories.items():
    agent_steps = [s for s in traj.steps if s.source == "agent"]
    tool_count = sum(len(s.tool_calls or []) for s in agent_steps)
    print(
        f"  {name}: {len(traj.steps)} steps, "
        f"{len(agent_steps)} agent turns, "
        f"{tool_count} tool calls, "
        f"{traj.final_metrics.total_prompt_tokens}/{traj.final_metrics.total_completion_tokens} tokens"
    )

## 4. Build `AtifEvalSample` Objects

Each sample pairs a trajectory with:
- **`output_obj`**: what the agent actually produced (extracted from the bash command that writes `result.json`)
- **`expected_output_obj`**: the ground truth from the BFCL benchmark

For BFCL tasks, the agent writes a JSON array to `/app/result.json` via a bash
command. We extract the agent's answer from the tool call arguments.

In [ ]:
import json
import re


def extract_bfcl_output(trajectory: ATIFTrajectory) -> str:
    """Extract the JSON written to /app/result.json from agent tool calls."""
    for step in trajectory.steps:
        if step.source != "agent" or not step.tool_calls:
            continue
        for tc in step.tool_calls:
            cmd = tc.arguments.get("command", "")
            if "/app/result.json" in cmd and "COMPLETE_TASK" not in cmd:
                match = re.search(r"echo\s+'?(.*?)'?\s*>\s*/app/result\.json", cmd)
                if match:
                    return match.group(1)
    return ""


for name, traj in trajectories.items():
    output = extract_bfcl_output(traj)
    print(f"  {name}: {output[:100]}")

In [ ]:
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvalSample

ground_truth = {
    "bfcl-live-simple": '[{"ChaFod": {"TheFod": "BURGER"}}]',
    "bfcl-live-irrelevance": '[]',
    "bfcl-live-multiple": (
        '[{"Travel_1_FindAttractions": '
        '{"location": "New York, NY", "free_entry": "True", '
        '"category": "Museum"}}]'
    ),
}

harbor_rewards = {
    "bfcl-live-simple": 1.0,
    "bfcl-live-irrelevance": 1.0,
    "bfcl-live-multiple": 0.0,
}

atif_samples = []
for name, traj in trajectories.items():
    agent_output = extract_bfcl_output(traj)
    sample = AtifEvalSample(
        item_id=name,
        trajectory=traj,
        expected_output_obj=ground_truth[name],
        output_obj=agent_output,
    )
    atif_samples.append(sample)
    print(
        f"  {name}:\n"
        f"    output:   {agent_output[:80]}\n"
        f"    expected: {ground_truth[name][:80]}\n"
        f"    harbor_reward: {harbor_rewards[name]}"
    )

print(f"\nCreated {len(atif_samples)} AtifEvalSample(s)")

## 5. Custom Evaluators (No API Keys Required)

We define two lightweight evaluators that run locally without any LLM calls.
These inherit from `AtifBaseEvaluator` and implement a single method:
`async def evaluate_atif_item(self, sample) -> EvalOutputItem`.

- **BFCLFunctionCallEvaluator**: Parses the JSON output and checks whether the
  agent called the correct function(s) with the right arguments.
- **TrajectoryEfficiencyEvaluator**: Scores based on how concisely the agent
  solved the task (fewer steps and tokens = higher score).

In [ ]:

from nat.data_models.evaluator import EvalOutputItem
from nat.plugins.eval.evaluator.atif_base_evaluator import AtifBaseEvaluator
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvalSample


class BFCLFunctionCallEvaluator(AtifBaseEvaluator):
    """Evaluates whether the agent called the correct function(s) with correct arguments.

    Scoring:
    - 1.0: function names AND arguments match expected output exactly
    - 0.5: function names match but arguments differ
    - 0.0: function names don't match or output is unparseable
    """

    def __init__(self, max_concurrency: int = 4) -> None:
        super().__init__(max_concurrency=max_concurrency)

    def _parse_bfcl_json(self, raw: str) -> list[dict] | None:
        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                return parsed
        except (json.JSONDecodeError, TypeError):
            pass
        return None

    def _extract_function_names(self, calls: list[dict]) -> set[str]:
        return {name for call in calls for name in call.keys()}

    async def evaluate_atif_item(self, sample: AtifEvalSample) -> EvalOutputItem:
        """Score one ATIF sample on function-call correctness."""
        expected = self._parse_bfcl_json(str(sample.expected_output_obj))
        generated = self._parse_bfcl_json(str(sample.output_obj))

        if expected is None or generated is None:
            return EvalOutputItem(
                id=sample.item_id,
                score=0.0,
                reasoning={"error": "Could not parse JSON", "raw_output": str(sample.output_obj)[:200]},
            )

        if expected == generated:
            score = 1.0
            detail = "exact_match"
        elif self._extract_function_names(expected) == self._extract_function_names(generated):
            score = 0.5
            detail = "function_names_match_args_differ"
        else:
            score = 0.0
            detail = "function_names_mismatch"

        return EvalOutputItem(
            id=sample.item_id,
            score=score,
            reasoning={
                "detail": detail,
                "expected_functions": sorted(self._extract_function_names(expected)) if expected else [],
                "generated_functions": sorted(self._extract_function_names(generated)) if generated else [],
                "expected": expected,
                "generated": generated,
            },
        )


class TrajectoryEfficiencyEvaluator(AtifBaseEvaluator):
    """Scores trajectory efficiency: fewer agent steps and tokens = higher score.

    Normalizes to [0, 1] using configurable upper bounds.
    """

    def __init__(
        self, max_steps: int = 10, max_tokens: int = 20000, max_concurrency: int = 4
    ) -> None:
        super().__init__(max_concurrency=max_concurrency)
        self._max_steps = max_steps
        self._max_tokens = max_tokens

    async def evaluate_atif_item(self, sample: AtifEvalSample) -> EvalOutputItem:
        """Score one ATIF sample on trajectory efficiency."""
        traj = sample.trajectory
        agent_steps = [s for s in traj.steps if s.source == "agent"]
        total_tokens = (
            (traj.final_metrics.total_prompt_tokens or 0)
            + (traj.final_metrics.total_completion_tokens or 0)
        ) if traj.final_metrics else 0

        step_score = max(0.0, 1.0 - len(agent_steps) / self._max_steps)
        token_score = max(0.0, 1.0 - total_tokens / self._max_tokens)
        score = round((step_score + token_score) / 2, 3)

        return EvalOutputItem(
            id=sample.item_id,
            score=score,
            reasoning={
                "agent_steps": len(agent_steps),
                "total_tokens": total_tokens,
                "step_score": round(step_score, 3),
                "token_score": round(token_score, 3),
            },
        )


print("Custom evaluators defined: BFCLFunctionCallEvaluator, TrajectoryEfficiencyEvaluator")

In [ ]:
from nat.plugins.eval.runtime.eval_harness import EvaluationHarness

harness = EvaluationHarness()

custom_results = await harness.evaluate(
    evaluators={
        "bfcl_function_call": BFCLFunctionCallEvaluator(),
        "trajectory_efficiency": TrajectoryEfficiencyEvaluator(),
    },
    atif_samples=atif_samples,
)

print("=" * 70)
print("Custom Evaluator Results")
print("=" * 70)
for eval_name, eval_output in custom_results.items():
    print(f"\n--- {eval_name} (avg={eval_output.average_score:.3f}) ---")
    for item in eval_output.eval_output_items:
        print(f"  {item.id}: score={item.score}")
        print(f"    {item.reasoning}")

### Compare with Harbor's Verifier

Harbor's own BFCL verifier assigned rewards. Let's compare our
`BFCLFunctionCallEvaluator` scores with Harbor's ground truth.

In [ ]:
fc_results = custom_results["bfcl_function_call"]

print(f"{'Task':<30} {'Harbor Reward':>15} {'NAT FC Score':>15}")
print("-" * 62)
for item in fc_results.eval_output_items:
    harbor = harbor_rewards[item.id]
    print(f"{item.id:<30} {harbor:>15.1f} {item.score:>15.1f}")

print("-" * 62)
print(
    f"{'Average':<30} "
    f"{sum(harbor_rewards.values()) / len(harbor_rewards):>15.3f} "
    f"{fc_results.average_score:>15.3f}"
)

## 6. RAGAS Evaluators (Requires `NVIDIA_API_KEY`)

For LLM-as-judge evaluation, we use the RAGAS `AnswerAccuracy` metric via the
`nvidia-nat-ragas` plugin. This requires an NVIDIA API key for NIM-hosted LLM
inference.

The same ATIF trajectories and `AtifEvalSample` objects are reused — only the
evaluator changes.

In [ ]:
import os

if "NVIDIA_API_KEY" not in os.environ:
    print(
        "NVIDIA_API_KEY not set — skipping RAGAS evaluation.\n"
        "Set it with: export NVIDIA_API_KEY='your-key-here'"
    )
    SKIP_RAGAS = True
else:
    SKIP_RAGAS = False
    print("NVIDIA_API_KEY is set. Proceeding with RAGAS evaluation.")

In [ ]:
if not SKIP_RAGAS:
    from nat.data_models.config import Config
    from nat.runtime.loader import PluginTypes
    from nat.runtime.loader import discover_and_register_plugins
    from nat.utils.data_models.schema_validator import validate_schema

    discover_and_register_plugins(PluginTypes.CONFIG_OBJECT)

    config_dict = {
        "llms": {
            "eval_llm": {
                "_type": "nim",
                "model_name": "nvidia/llama-3.3-nemotron-super-49b-v1",
                "temperature": 0.0,
            },
        },
        "eval": {
            "general": {"max_concurrency": 1},
            "evaluators": {
                "accuracy": {
                    "_type": "ragas",
                    "llm_name": "eval_llm",
                    "metric": "AnswerAccuracy",
                    "enable_atif_evaluator": True,
                },
            },
        },
    }

    config = validate_schema(config_dict, Config)
    print(f"Evaluators configured: {list(config.eval.evaluators.keys())}")

In [ ]:
if not SKIP_RAGAS:
    from nat.plugins.eval.evaluator.atif_evaluator import AtifEvaluator
    from nat.plugins.eval.runtime.builder import WorkflowEvalBuilder

    eval_builder = WorkflowEvalBuilder(
        general_config=config.general,
        eval_general_config=config.eval.general,
    )
    await eval_builder.__aenter__()
    await eval_builder.populate_builder(config, skip_workflow=True)

    atif_evaluators = {}
    for name in config.eval.evaluators:
        evaluator_info = eval_builder.get_evaluator(name)
        if isinstance(evaluator_info, AtifEvaluator):
            atif_evaluators[name] = evaluator_info
            print(f"  [ATIF] {name}: {evaluator_info.description}")

    ragas_results = await harness.evaluate(
        evaluators=atif_evaluators,
        atif_samples=atif_samples,
    )

    await eval_builder.__aexit__(None, None, None)

    print("\n" + "=" * 70)
    print("RAGAS Evaluation Results")
    print("=" * 70)
    for eval_name, eval_output in ragas_results.items():
        print(f"\n--- {eval_name} (avg={eval_output.average_score:.3f}) ---")
        for item in eval_output.eval_output_items:
            print(f"  {item.id}: score={item.score}")
            if item.reasoning:
                reasoning_str = str(item.reasoning)
                print(f"    reasoning: {reasoning_str[:200]}{'...' if len(reasoning_str) > 200 else ''}")

## 7. Summary

This notebook demonstrated end-to-end interoperability between a third-party
agent framework (Harbor) and the NeMo Agent Toolkit evaluation system:

1. **Harbor generates ATIF trajectories** — `mini-swe-agent` ran BFCL
   function-calling tasks and produced standard ATIF JSON output.

2. **NeMo Agent Toolkit parses and validates** — `ATIFTrajectory.model_validate()`
   loaded the Harbor output into typed Pydantic models.

3. **Custom evaluators scored without LLMs** — `BFCLFunctionCallEvaluator`
   checked function-call correctness; `TrajectoryEfficiencyEvaluator` measured
   step and token efficiency. No API keys needed.

4. **RAGAS evaluators scored with LLM-as-judge** — The same `AtifEvalSample`
   objects were evaluated by RAGAS `AnswerAccuracy` using NIM endpoints.

| Component | What We Used |
|---|---|
| **Agent framework** | Harbor (`mini-swe-agent`) |
| **Benchmark** | BFCL (Berkeley Function-Calling Leaderboard) |
| **Model** | `meta/llama-3.3-70b-instruct` (via NVIDIA NIM) |
| **Interchange format** | ATIF v1.2 |
| **Eval harness** | `nvidia-nat-eval` (`EvaluationHarness`) |
| **Custom evaluators** | `BFCLFunctionCallEvaluator`, `TrajectoryEfficiencyEvaluator` |
| **LLM evaluators** | RAGAS `AnswerAccuracy` via `nvidia-nat-ragas` |

Any agent framework that outputs ATIF can be evaluated this way — the eval
harness is agnostic to how the trajectory was produced.